# Расчёт метрик качества моделей сегментации.

## Методика расчёта

При оценке качества моделей для задач бинарной и многоклассовой сегментации используются значения каждого пикселя.

**Confusion matrix (матрица ошибок)**

 |     |$y = 1$ | $y = 0$ |
 | --- | :----: | :-----: |
| $\hat y = 1$  |	True Positive (TP)  |	False Positive (FP) |
| $\hat y = 0$  |	False Negative (FN)  |	True Negative (TN) |

Здесь $\hat y$ — это предсказание модели для пикселя, а $y$ — истинная метка класса на этом пикселе.

**Precision и recall**

*Precision* можно интерпретировать как долю пикселей, названных классификатором положительными и при этом действительно являющимися положительными, а *recall* показывает, какую долю пикселей положительного класса из всех объектов положительного класса нашел алгоритм.
Положительным классов в нашем случае является целевой.


$\large precision = \frac{TP}{TP + FP}$


$\large recall = \frac{TP}{TP + FN}$

## F1-score

F-мера (в общем случае $\ F_\beta$) — среднее гармоническое precision и recall :

$\large \ F_\beta = (1 + \beta^2) \cdot \frac{precision \cdot recall}{(\beta^2 \cdot precision) + recall}$

$\beta$ в данном случае определяет вес точности в метрике.

В нашем случае при расчёте F1 $\beta = 1$ это среднее гармоническое.


In [1]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import Polygon


from aeronet import dataset as ds

In [2]:
import warnings
warnings.filterwarnings("ignore")


# Folder structure:

`root \ [task] \ [sample] \ [gt.geojson/features.geojson/image.tif]`

- task - name of the problem solved (roads, parcels, etc.)
  - sample - name of sample of the dataset. There should be no other folders in the [task] dir. Name can be any
    - gt.geojson - ground truth file
    - features.geojson - prediction file
    - image.tif - source image file (for display purposes only)



# Define metrics calculation function

In [3]:
AREA_TP = 'AREA_TP'
AREA_FP = 'AREA_FP'
AREA_FN = 'AREA_FN'
AREA_IOU = 'AREA_IOU'
AREA_PRECISION = 'AREA_PRECISION'
AREA_RECALL = 'AREA_RECALL'
AREA_F1 = 'AREA_F1'

OBJ_TP = 'OBJ_TP'
OBJ_FP = 'OBJ_FP'
OBJ_FN = 'OBJ_FN'
OBJ_PRECISION = 'OBJ_PRECISION'
OBJ_RECALL = 'OBJ_RECALL'
OBJ_F1 = 'OBJ_F1'


class VectorMetricsCalculator(object):

    def __init__(self, pd_fc, gt_fc, obj_iou_threshold=0.5):
        self.pd_fc = pd_fc
        self.gt_fc = gt_fc
        self.obj_iou_threshold = obj_iou_threshold
        self.metrics_cache = {}

        # Declare available metrics
        self._name_to_func = {
            AREA_TP: self.get_area_iou_f1,
            AREA_FP: self.get_area_iou_f1,
            AREA_FN: self.get_area_iou_f1,
            AREA_IOU: self.get_area_iou_f1,
            AREA_PRECISION: self.get_area_iou_f1,
            AREA_RECALL: self.get_area_iou_f1,
            AREA_F1: self.get_area_iou_f1,

            OBJ_TP: self.get_obj_precision_recall_f1,
            OBJ_FP: self.get_obj_precision_recall_f1,
            OBJ_FN: self.get_obj_precision_recall_f1,
            OBJ_PRECISION: self.get_obj_precision_recall_f1,
            OBJ_RECALL: self.get_obj_precision_recall_f1,
            OBJ_F1: self.get_obj_precision_recall_f1,
        }

    def available_metrics(self):
        return [i for i in self._name_to_func.keys()]

    def by_name(self, name):
        if name not in self.metrics_cache:
            self.metrics_cache.update(self._name_to_func[name]())
        return self.metrics_cache[name]

    def get_names(self):
        return [k for k in self._name_to_func.keys()]

    def calc_f1(self, tp, fp, fn):
        if tp == 0:
            return 0
        return float(tp / (tp + 0.5 * (fp + fn)))

    def calc_recall(self, tp, fn):
        if tp == 0:
            return 0
        return float(tp / (tp + fn))

    def calc_precision(self, tp, fp):
        if tp == 0:
            return 0
        return float(tp / (tp + fp))

    def calc_iou_feat(self, pr_feature, gt_feature):
        intersection = gt_feature.intersection(pr_feature).area
        union = gt_feature.union(pr_feature).area

        if intersection == 0:
            return 0

        return float(intersection / union)

    def calc_iou_obj(self, pd_fc, gt_fc):
        scores = []
        for feature in gt_fc:
            proposed_features = pd_fc.intersection(feature)
            feature_scores = [self.calc_iou_feat(f, feature) for f in proposed_features]
            if feature_scores:
                max_iou_score = max(feature_scores)
            else:
                max_iou_score = 0
            scores.append(max_iou_score)

        return np.array(scores)

    def get_obj_precision_recall_f1(self):
        pd_fc = self.pd_fc
        gt_fc = self.gt_fc
        iou_threshold = self.obj_iou_threshold

        iou_obj_scores = self.calc_iou_obj(pd_fc, gt_fc)
        tp = int(sum(iou_obj_scores > iou_threshold))
        fp = int(len(pd_fc) - tp)
        fn = int(len(gt_fc) - tp)

        return {
            OBJ_TP: tp,
            OBJ_FP: fp,
            OBJ_FN: fn,
            OBJ_PRECISION: self.calc_precision(tp, fp),
            OBJ_RECALL: self.calc_recall(tp, fn),
            OBJ_F1: self.calc_f1(tp, fp, fn)
        }

    def get_area_iou_f1(self):
        pd_fc = self.pd_fc
        gt_fc = self.gt_fc

        tp = 0
        fn = 0
        for gt_f in gt_fc:
            gt_f_area = gt_f.area
            for pd_f in pd_fc.intersection(gt_f):
                intersection_area = gt_f.intersection(pd_f).area
                if intersection_area > 0:
                    tp += intersection_area
                    gt_f_area -= intersection_area
            fn += gt_f_area

        fp = 0
        for pd_f in pd_fc:
            pd_f_area = pd_f.area
            for gt_f in gt_fc.intersection(pd_f):
                intersection_area = gt_f.intersection(pd_f).area
                if intersection_area > 0:
                    pd_f_area -= intersection_area
            fp += pd_f_area

        tp = float(tp)
        fp = float(fp)
        fn = float(fn)

        n = float(tp)
        d = float(tp + fp + fn + 1e-12)

        iou = float(np.divide(n, d))
        f1 = float(self.calc_f1(tp, fp, fn))

        precision = self.calc_precision(tp, fp)
        recall = self.calc_recall(tp, fn)

        return {
            AREA_TP: tp,
            AREA_FP: fp,
            AREA_FN: fn,
            AREA_IOU: iou,
            AREA_PRECISION: precision,
            AREA_RECALL: recall,
            AREA_F1: f1,
        }


def read_fc(path, name, crs=None):
    fp = os.path.join(path, '{}.geojson'.format(name))
    fc = ds.FeatureCollection.read(fp)

    if crs == 'utm':
        fc = fc.reproject_to_utm()
    elif crs is not None:
        fc = fc.reproject(crs)
    return fc

## Cut ground truth files to images

In [4]:
def get_image_extent_polygon(image_file):
    with rasterio.open(image_file) as src:
        # BoundingBox(left=358485.0, bottom=4028985.0, right=590415.0, top=4265115.0)
        extent = src.bounds
        crs = src.crs
    #  classmethod from_bounds(xmin, ymin, xmax, ymax)
    geom = Polygon.from_bounds(*extent)
    #src_crs, dst_crs, geom, antimeridian_cutting=True, antimeridian_offset=10.0, precision=-1)
    feat = ds.Feature(geom, crs=crs)
    feat = feat.reproject("EPSG:4326")
    return feat

def cut_gt_to_image(full_gt, image_file, out_file):
    extent = get_image_extent_polygon(image_file)
    cut_gt = full_gt.intersection(extent)
    cut_gt.save(out_file)

def cut_gt(gt_file, folders, overwrite=True):
    full_gt = ds.FeatureCollection.read(gt_file).reproject("EPSG:4326")
    for folder in folders:
        image_files = [entry for entry in folder.iterdir() if entry.suffix.lower() in ['.tif', '.tiff']]
        if not image_files:
            print(f"File {image_file} does not exist!")
            continue
        image_file = image_files[0]
        out_file = image_file.with_name('gt.geojson')
        if os.path.exists(out_file) and not overwrite:
            print(f"GT cut file {out_file} already existst!")
            continue
        cut_gt_to_image(full_gt, image_file, out_file)


## Visualize

In [5]:
def visualize_files(folder, image_name='image', gt_name='gt', pred_name='features'):
    with rasterio.open(folder/(image_name + '.tif')) as src:
        transform = src.transform
        shape = src.shape
        rgb = src.read()
        crs = src.crs

    gt_fc = read_fc(folder, gt_name, crs=crs)
    pred_fc = read_fc(folder, pred_name, crs=crs)
    gt_mask = ds.rasterize(gt_fc,
                          transform = transform,
                          out_shape=shape,
                          name='gt_mask').numpy()

    pred_mask = ds.rasterize(pred_fc,
                          transform = transform,
                          out_shape=shape,
                          name='pred_mask').numpy()

    tp_mask = gt_mask * pred_mask
    tn_mask = (1-gt_mask)*(1-pred_mask)
    fp_mask = pred_mask*(1-gt_mask)
    fn_mask = gt_mask*(1-pred_mask)

    masks = np.stack((fp_mask*255, tp_mask*255, fn_mask*255), axis=-1)
    plt.figure(figsize=(30,10))
    plt.suptitle(f"Sample {folder.name}", fontsize=36)
    plt.subplot(131)
    plt.imshow(np.transpose(rgb, (1, 2, 0)))
    plt.subplot(132)
    plt.imshow(masks)
    plt.subplot(133)
    TP = tp_mask.sum()
    FP = fp_mask.sum()
    FN = fn_mask.sum()
    
    text = f"TP [GREEN]= {TP} \n" \
          f"FP [RED]= {FP} \n" \
          f"FN [BLUE]= {FN} \n" \
          f"TN [BLACK]= {tn_mask.sum()} "
    plt.text( 0.05, 0.3, text, fontsize=36)
    plt.show()
    
    F1 = TP/(TP + 0.5*(FP + FN))
    return {
        "AREA_F1": F1,
        "AREA_TP": TP,
        "AREA_FP": FP,
        "AREA_FN": FN
    }

## Calculate for every sample

In [6]:
def calc_metric_sample(folder, visualize=False, verbose=False):
    result_dict = visualize_files(folder, get_image(folder).stem, 'gt', 'features')
    if verbose:
        print(f"Sample {folder.name}: "
          f"F1 = {result_dict['AREA_F1']}, TP = {result_dict['AREA_TP']}, FP = {result_dict['AREA_FP']}, FN = {result_dict['AREA_FN']}")
    return result_dict

def get_image(folder: Path):
    image_files = [entry for entry in folder.iterdir() if entry.suffix.lower() in ['.tif', '.tiff']]
    if not image_files:
        return None
    return image_files[0]

def calc_metric_task(task: str, root: Path, visualize=False, verbose=False):
    
    folder = root/task
    subfolders = [folder for folder in folder.iterdir() if folder.is_dir() and (folder/'features.geojson').exists() and get_image(folder)]
    print(f"Got {len(subfolders)} data samples")
    print("Preparing ground truth data - divide by folders")
    cut_gt(root/f"test_{task}.geojson", subfolders, overwrite=True)
    
    print("Calculating metrics")
    all_metrics = [calc_metric_sample(sample, visualize=visualize, verbose=verbose) for sample in subfolders]

    tp = fp = fn = tn = 0

    # For the proper averaging, we need to rely on samples' statistics
    # and calculate F1-score after the statistics summation
    for sample in all_metrics:
        tp += sample[AREA_TP]
        fp += sample[AREA_FP]
        fn += sample[AREA_FN]

    if verbose:
        print(f"Total: TP = {tp}, FP = {fp}, FN = {fn}")
    f1_score = (tp / (tp + 0.5 * (fp + fn)))
    return {folder.name : f1_score}

def calc_metrics_all(root: Path, visualize=False):
    result = {}
    for task in tasks:
        result.update(calc_metric_task(task, root, visualize=visualize))
    return result


# Run metrics calculation and visualization

## Task: roads

In [ ]:
rootdir = Path("/home/user/data")
task = 'road'
result = calc_metric_task(task, root=rootdir, visualize=True, verbose=True)

print(f"Average F1 Score for {task} = {result[task]}")